# Data budget

A worked example of the `quicksat` data and downlink budget for the sample satellite. It finds how much data the payload generates, and how much the downlink sends. Then it checks whether the downlink keeps up, and whether the mass memory holds the data between contacts.

`docs/sizing/data_budget_ref.ipynb` explains *why* each step works the way it does. It covers the byte convention, the unit errors to avoid, and why the input is a config file rather than a CSV. `docs/sizing/mission_ref.ipynb` covers the mission that the budget uses.

In [1]:
# Change log. LAST_CHANGE and CHANGE_NOTE are typed in by hand: update them whenever
# the inputs move -- a duty cycle revised, a contact count renegotiated -- so that the
# figures below can be read against what produced them. The run time is recorded
# automatically, and says how stale the outputs stored in this notebook are relative
# to that last change.
from datetime import datetime

LAST_CHANGE = "2026-10-21"
CHANGE_NOTE = "The 2 TB mass memory is in the data file; the 3 orbits without contact are a requirement."

print(f"last change  {LAST_CHANGE}")
print(f"             {CHANGE_NOTE}")
print(f"last run     {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}")

last change  2026-10-21
             The 2 TB mass memory is in the data file; the 3 orbits without contact are a requirement.
last run     2026-10-21 23:31 CEST


In [2]:
import sys
from pathlib import Path

import pandas as pd

# quicksat is installed with pip. In a clone of the repository, use the package
# beside sample/ instead, so that it runs without installing. Either way the
# notebook runs in its own folder, and reads data/ beside it.
repo = Path.cwd().parents[1]
if (repo / "quicksat" / "__init__.py").exists():
    sys.path.insert(0, str(repo))

from quicksat.dataflow.budget import DataBudget, DataFlowModel
from quicksat.utils.checks import tabulate as tabulate_checks
from quicksat.utils.mission import Mission
from quicksat.utils.requirements import Requirements

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## The inputs

There are three files: the shared mission, the shared requirements, and the payload dataflow model. The mission and the requirements are loaded separately, because the other budgets read the same ones. Loading each once here and passing it on keeps a single object, rather than one copy per budget. Where nothing else needs them, `DataBudget.from_yaml_file()` takes the three paths and reads the files itself.

The dataflow model holds what the spacecraft is: the payload, the link and the mass memory's capacity. The requirements hold what the mission asks of it: here, the longest run of orbits with no usable contact.

In [3]:
DATA = Path("data")
mission = Mission.from_yaml_file(DATA / "mission.yaml")
requirements = Requirements.from_yaml_file(DATA / "requirements.yaml")
model = DataFlowModel.from_yaml_file(DATA / "pl_dataflow_model.yaml")

data_budget = DataBudget(model, mission, requirements)

In [4]:
gen = data_budget.model.generation
dl = data_budget.model.downlink

print(f"mission      {mission.altitude:.0f} circular, {mission.inclination}")
print(f"             period {mission.period:.0f}, {mission.orbits_per_day:.3f} orbits/day")
print(f"generation   {gen.raw_datarate:.0f} raw, {gen.compression_ratio}x compression,"
      f" {gen.duty_cycle} duty cycle")
print(f"downlink     {dl.rate:.0f}, {dl.contacts_per_day:g} contacts/day"
      f" at {dl.avg_contact_duration:.0f}")
print(f"storage      {model.storage.capacity} mass memory, for"
      f" {data_budget.required_orbits:g} orbits without contact")

mission      500 km circular, 97.4 deg
             period 5677 s, 15.219 orbits/day
generation   1000 Mbit / s raw, 2.4x compression, 5.0 % duty cycle
downlink     800 Mbit / s, 7 contacts/day at 6 min
storage      2.0 Tbyte mass memory, for 3 orbits without contact


## Generation and downlink

Generation is counted per orbit, because the payload collects data for a fraction of each orbit. Downlink is counted per day, because contacts follow the ground station's day. The two are compared per day.

In [5]:
print(f"effective rate    {data_budget.effective_datarate:.2f}  "
      f"({gen.raw_datarate:.0f} raw / {gen.compression_ratio}x)")
print(f"collecting for    {data_budget.generation_duration:.1f} of every {mission.period:.0f}")
print()
print(f"generated         {data_budget.generated_per_orbit:.2f} per orbit")
print(f"                  {data_budget.generated_per_day:.2f} per day")
print()
print(f"contacts          {dl.contacts_per_day:g} per day at {dl.avg_contact_duration:.0f}"
      f"  =  {data_budget.contact_per_day:.0f} per day")
print(f"downlinked        {data_budget.downlinked_per_day:.2f} per day")
print(f"                  {data_budget.downlinked_per_orbit:.2f} per orbit")

effective rate    416.67 Mbit / s  (1000 Mbit / s raw / 2.4x)
collecting for    283.8 s of every 5677 s

generated         14.78 Gbyte per orbit
                  225.00 Gbyte per day

contacts          7 per day at 6 min  =  42 min per day
downlinked        252.00 Gbyte per day
                  16.56 Gbyte per orbit


## Does it close?

This is the one number that the budget exists to produce.

In [6]:
margin = data_budget.margin.value
balance = data_budget.downlinked_per_day - data_budget.generated_per_day

print(f"generated    {data_budget.generated_per_day:.2f} per day")
print(f"downlinked   {data_budget.downlinked_per_day:.2f} per day")
print(f"margin       {margin * 100:+.1f}%")
print()
if margin >= 0:
    print(f"CLOSES with {balance:.2f} per day to spare")
else:
    print(f"DOES NOT CLOSE - short by {abs(balance):.2f} per day")
    print(f"needs {1 / (1 + margin):.2f}x the current downlink capacity")

generated    225.00 Gbyte per day
downlinked   252.00 Gbyte per day
margin       +12.0%

CLOSES with 27.00 Gbyte per day to spare


### How much headroom is there?

A margin is only as good as its assumptions, and every input here is an estimate. The useful question is not "what is the margin", but "how far can each input change before the margin is gone".

Each row below is the **break-even value** for one input, with every other input as the files give it. For example, take the duty cycle row. The payload could collect data for up to that fraction of each orbit before the downlink stops keeping up.

In [7]:
headroom = 1 + margin

pd.DataFrame(
    [
        ("raw data rate", f"{gen.raw_datarate:.0f}",
         f"{(gen.raw_datarate * headroom).to('Mbit/s'):.0f}", "generate faster"),
        ("duty cycle", f"{gen.duty_cycle}",
         f"{gen.duty_cycle * headroom:.2f}", "collect longer"),
        ("compression ratio", f"{gen.compression_ratio}x",
         f"{gen.compression_ratio / headroom:.2f}x", "compress less"),
        ("contacts per day", f"{dl.contacts_per_day:g}",
         f"{dl.contacts_per_day / headroom:.2f}", "fewer passes"),
        ("contact duration", f"{dl.avg_contact_duration:.0f}",
         f"{(dl.avg_contact_duration / headroom).to('min'):.2f}", "shorter passes"),
        ("downlink rate", f"{dl.rate:.0f}",
         f"{(dl.rate / headroom).to('Mbit/s'):.0f}", "slower link"),
    ],
    columns=["input", "now", "break-even", "moving which way"],
)

,input,now,break-even,moving which way
0,raw data rate,1000 Mbit / s,1120 Mbit / s,generate faster
1,duty cycle,5.0 %,5.60 %,collect longer
2,compression ratio,2.4x,2.14x,compress less
3,contacts per day,7,6.25,fewer passes
4,contact duration,6 min,5.36 min,shorter passes
5,downlink rate,800 Mbit / s,714 Mbit / s,slower link


## Storage

The memory needed over a run of orbits with no usable contact. The requirements file sets the run, and the dataflow model gives the memory's capacity. `storage_required()` takes the requirement's run by default, and any other run as an argument.

In [8]:
orbits = data_budget.required_orbits
print(f"{orbits:g} orbits without contact  ->  {data_budget.storage_required():.2f}   (the requirement)")
print(f"1 orbit                   ->  {data_budget.storage_required(1):.2f}")
print(f"a full day                ->  {data_budget.storage_required(mission.orbits_per_day.value):.2f}")
print()
print(f"mass memory               {model.storage.capacity.to('GB'):.0f}")

3 orbits without contact  ->  44.35 Gbyte   (the requirement)
1 orbit                   ->  14.78 Gbyte
a full day                ->  225.00 Gbyte

mass memory               2000 Gbyte


## The checks

`checks()` gives the budget's checks. The downlink must send the data generated each day, which is the margin above. The mass memory must hold the data of the required run without contact. Each check compares a figure of the design with its limit, and gives the spare and the margin.

`tabulate()` from `quicksat.utils.checks` lays the checks out. The Sizing Editor shows the same checks, from the same code.

In [9]:
tabulate_checks(data_budget.checks())

check,description,design,limit,spare,margin,verdict
downlink,The downlink sends the data generated each day,252.00 GB,≥ 225.00 GB,+27.00 GB,+12.0 %,PASS
storage,The mass memory holds the data of 3 orbits without contact,2000.00 GB,≥ 44.35 GB,+1955.65 GB,+4409.5 %,PASS


## The budget, in one table

`tabulated_data()` lays out the whole chain as a document. Each row gives the item, its value, its unit, and the assumption behind it. The mass memory comes last, with the storage required when the budget has a data requirement. The frame is available as `.data`, with every row tagged `input`, `derived`, `margin` or `storage`.

In [10]:
data_budget.tabulated_data()

,Value,Unit,Comment
Orbital period,94.62,min,500 km circular
Orbits per day,15.22,-,
Data generation rate,416.67,Mbit/s,"1000 Mbit / s raw, 2.4x compression"
Data generation duration,283.85,s/orbit,5 % duty cycle
Data generated,14.78,GB/orbit,
,225.00,GB/day,
Data downlink rate,800.00,Mbit/s,achieved throughput
Contacts,7.00,per day,6 min average duration
Data downlink duration,42.00,min/day,
Data downlinked,16.56,GB/orbit,
